# Figure 4: six-condition comparison

A: condition waveform icons. B: FFT per condition. C: randomized trial order. D–F: total power, flutter-band power, % reduction.

In [ ]:
import glob, os
import cairosvg
import figurefirst as fifi
from IPython.display import display, SVG

from flutter.plotting import set_style
from flutter.config import OL_RDM_DIR, OUTPUT_DIR
from flutter.condition_icons import draw_condition_icons_row, ICON_CONDITIONS
from flutter.trial_order import collect_trial_order, draw_trial_order_panel
from flutter.fft_energy import (process_condition_fft, draw_single_condition_fft_overlay,
                                add_panel_label, CONDITIONS, CONDITION_LABELS, CONDITION_COLORS)
from flutter.frequency_profile import draw_frequency_triptych_panel
from flutter.condition_comparison import load_all_conditions

set_style()

In [ ]:
layout = fifi.svg_to_axes.FigureLayout('fig4.svg', autogenlayers=True, make_mplfigures=True, hide_layers=[])
print(layout.axes.keys())
print(layout.figures.keys())

In [ ]:
CONDITION_BAG_PATTERNS = {
    'CLOSED_LOOP':      'CLOSED_LOOP_sample*.bag',
    'OPEN_LOOP_55':     'OPEN_LOOP_55_sample*.bag',
    'POISSON':          'POISSON_sample*.bag',
    'UNIFORM_JITTER':   'UNIFORM_JITTER_sample*.bag',
    'PHASE_RANDOMIZED': 'PHASE_RANDOMIZED_sample*.bag',
    'REPLAY':           'REPLAY_sample*.bag',
}

all_results = {}
for condition, pattern in CONDITION_BAG_PATTERNS.items():
    bag_paths = sorted(glob.glob(os.path.join(str(OL_RDM_DIR), pattern)))
    if bag_paths:
        all_results[condition] = process_condition_fft(bag_paths, is_closed_loop=(condition == 'CLOSED_LOOP'))

In [ ]:
trials = collect_trial_order(str(OL_RDM_DIR))

In [ ]:
# Per-sample JSON reductions. Not drawn in any current panel; kept in case it's needed.
ol_rdm_conditions = load_all_conditions(OL_RDM_DIR)

In [ ]:
# Panel A: condition icons
icon_axes = [
    layout.axes[('fig4', 'icon_CL')],
    layout.axes[('fig4', 'icon_OL')],
    layout.axes[('fig4', 'icon_Poisson')],
    layout.axes[('fig4', 'icon_Jitter')],
    layout.axes[('fig4', 'icon_Polarity')],
    layout.axes[('fig4', 'icon_Replay')],
]
draw_condition_icons_row(icon_axes, ICON_CONDITIONS, scale=0.7)

In [ ]:
# Panel C: trial order
draw_trial_order_panel(layout.axes[('fig4', 'Trial_order')], trials, scale=0.7, show_legend=False)

In [ ]:
# Panel B: FFT row
fft_axes = {
    'CLOSED_LOOP':      layout.axes[('fig4', 'FFT_CL')],
    'OPEN_LOOP_55':     layout.axes[('fig4', 'FFT_OL')],
    'POISSON':          layout.axes[('fig4', 'FFT_Poisson')],
    'UNIFORM_JITTER':   layout.axes[('fig4', 'FFT_Jitter')],
    'PHASE_RANDOMIZED': layout.axes[('fig4', 'FFT_Polarity')],
    'REPLAY':           layout.axes[('fig4', 'FFT_Replay')],
}
center_condition = 'CLOSED_LOOP'

for condition, ax in fft_axes.items():
    draw_single_condition_fft_overlay(ax, all_results, condition, scale=0.6,
                                      f_max=19, show_legend=False, show_title=False,
                                      show_inset=True, inset_freq_range=(4.0, 6.0),
                                      inset_bbox=(0.52, 0.52, 0.44, 0.44))
    if condition != center_condition:
        ax.set_xlabel('')
    ax.tick_params(axis='y', labelsize=5)
    if condition != 'CLOSED_LOOP':
        ax.set_ylabel('')
        ax.tick_params(axis='y', labelleft=False)

In [ ]:
# Panels D-F: same triptych function as fig3, with the six conditions
SHORT_LABELS = {
    'CLOSED_LOOP':      'CL',
    'OPEN_LOOP_55':     'OL',
    'POISSON':          'P',
    'UNIFORM_JITTER':   'UJ',
    'PHASE_RANDOMIZED': 'PR',
    'REPLAY':           'R',
}
SIX_CONDITIONS = [(k, SHORT_LABELS.get(k, CONDITION_LABELS[k]), CONDITION_COLORS[k])
                  for k in CONDITIONS if k in all_results]

draw_frequency_triptych_panel(layout.axes[('fig4', 'total_power')], all_results,
                              SIX_CONDITIONS, metric='energy_total', scale=0.7, xtick_fontsize=15)
draw_frequency_triptych_panel(layout.axes[('fig4', 'flutter_power')], all_results,
                              SIX_CONDITIONS, metric='energy_flutter', scale=0.7, xtick_fontsize=11)
draw_frequency_triptych_panel(layout.axes[('fig4', 'flutter_percent_reduc')], all_results,
                              SIX_CONDITIONS, metric='pct_reduction', scale=0.7, xtick_fontsize=11)

In [ ]:
add_panel_label(icon_axes[0], 'A', y=1.25, x=-0.14)
add_panel_label(fft_axes['CLOSED_LOOP'], 'B', y=1.25, x=-0.14)
add_panel_label(layout.axes[('fig4', 'Trial_order')], 'C', y=1.9, x=0.01)
add_panel_label(layout.axes[('fig4', 'total_power')], 'D')
add_panel_label(layout.axes[('fig4', 'flutter_power')], 'E')
add_panel_label(layout.axes[('fig4', 'flutter_percent_reduc')], 'F', x=-0.022)

In [ ]:
layout.append_figure_to_layer(layout.figures['none'], 'none', cleartarget=True)
out_svg = OUTPUT_DIR / 'fig4_filled.svg'
layout.write_svg(str(out_svg))
display(SVG(str(out_svg)))

In [ ]:
cairosvg.svg2pdf(url=str(out_svg), write_to=str(OUTPUT_DIR / 'fig4_filled.pdf'))